# Projet complet — Attrition synthétique

Question : prédire un label binaire simulé à partir d’informations disponibles aujourd’hui. Ce corpus synthétique n’est pas une mesure du comportement de vrais abonnés. Durée : 2–3 h.

## 1. Explorer les lignes et la cible

Les labels sont tirés avec du bruit à partir d’ancienneté, tarif, appels et contrat. Il n’existe pas de colonne cible cachée dans `X`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from learning_hub.preprocessing import make_churn_data

features, target = make_churn_data(n_samples=1_000, random_state=31)
data = features.assign(churn=target)
print(data.head()); print(data.dtypes); print(data.isna().sum())
print('proportion positive:', round(data.churn.mean(), 3))
print(data.groupby('churn')[['tenure_months', 'monthly_charge', 'support_calls_90d']].mean().round(1))

## 2. Baseline et split stratifié

La classe majoritaire est un niveau de comparaison. Les données sont séparées avant que l’imputeur, scaler ou encodeur apprenne quoi que ce soit.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, precision_recall_curve, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from learning_hub.preprocessing import make_tabular_preprocessor

x_train, x_test, y_train, y_test = train_test_split(features, target, stratify=target, test_size=0.25, random_state=31)
baseline = DummyClassifier(strategy='most_frequent').fit(x_train, y_train)
pipeline = Pipeline([('prep', make_tabular_preprocessor(['tenure_months','monthly_charge','support_calls_90d'], ['contract'])), ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))]).fit(x_train, y_train)
probability = pipeline.predict_proba(x_test)[:, 1]
prediction = (probability >= 0.5).astype(int)
print('baseline accuracy:', round(baseline.score(x_test, y_test), 3))
print('Average Precision:', round(average_precision_score(y_test, probability), 3))
print('ROC-AUC:', round(roc_auc_score(y_test, probability), 3))
print(classification_report(y_test, prediction, zero_division=0))

## 3. Erreurs, seuil et conséquences

La matrice de confusion et la PR curve montrent les compromis du seuil. Un changement de seuil doit suivre une conséquence d’usage définie, et se choisir dans la validation, pas sur le test final.

In [ ]:
ConfusionMatrixDisplay(confusion_matrix(y_test, prediction), display_labels=['retained','churn']).plot(cmap='Blues', values_format='d'); plt.show()
precision, recall, thresholds = precision_recall_curve(y_test, probability)
plt.plot(recall, precision); plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Test PR curve (descriptive only)'); plt.show()
errors = x_test.copy(); errors['actual'] = y_test; errors['predicted'] = prediction; errors['probability'] = probability
errors.loc[errors.actual != errors.predicted].head(10)

## 4. Conclusion, risques et suite

La classification synthétique permet de vérifier les shapes, le split stratifié, le preprocessing, les scores et l’analyse d’erreurs. Les métriques décrivent uniquement ce générateur et ce split.

Avant toute application réelle, définir consentement et minimisation des données, vérifier représentativité et biais, mesurer calibration, fixer une règle de décision avec les parties prenantes, puis prévoir feedback différé et monitoring. Ne pas agir sur des personnes avec ce modèle jouet.